# FINAL RESULTS — MorphoWaveNet vs purane ECG models (PTB-XL + Chapman)

**Yeh notebook kya karta hai:** koi training nahi. Pehle se train kiye hue saare models (`.pt` files) load karta hai, sabko **ek hi protocol** pe evaluate karta hai, aur sir ko dikhane wale saare tables + graphs `results_final/` folder mein save karta hai.

**Task:** PTB-XL superdiagnostic, 5 classes (NORM, MI, STTC, CD, HYP), multi-label. Split: folds 1–8 train, fold 9 val (sirf thresholds ke liye), fold 10 test. External test: Chapman-Shaoxing (koi training nahi).

**Run kaise karein:** yeh file `coding/` folder mein rakho (jahan `processed_data/`, `processed_chapman/` aur `.pt` files hain) → **Run All**. GPU pe ~2–5 min, CPU pe ~15–20 min. Predictions cache ho jaati hain, dobara run instant hoga.

| # | Result | Kya dikhata hai |
|---|---|---|
| 1 | PTB-XL test table | Saare models ka AUROC / AUPRC / F1 + 95% CI |
| 2 | Published papers se comparison | Strodthoff 2021 (IEEE JBHI), ECGFounder 2026 etc. vs hamara model |
| 3 | Statistical test | Kya improvement sach mein significant hai (paired bootstrap) |
| 4 | Per-class AUROC | Kis bimari mein kitna fayda |
| 5 | Chapman external validation | Doosre hospital/machine pe performance |
| 6 | Efficiency | Parameters vs AUROC |
| 7 | Explainability | Model ne kaunsi lead aur ECG ka kaunsa hissa dekha |
| 8 | Problem → Solution → Evidence | Purane models ki problem aur hamara solution, numbers ke saath |

In [ ]:
# ======================= CONFIG =======================
import os, json, time, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

PTB_DIR   = Path('processed_data')        # X_train/val/test.npy (N,1000,12) + y_*.npy  (CNN_base_model.ipynb se bana)
CHAP_DIR  = Path('processed_chapman')     # X_chapman.npy (N,12,1000) + y_chapman.npy (chapman_dataset.ipynb se bana)
CKPT_DIR  = Path('.')                     # saari .pt files yahin hain
OUT       = Path('results_final'); (OUT / 'preds').mkdir(parents=True, exist_ok=True)

N_BOOT      = 1000     # PTB-XL bootstrap resamples
N_BOOT_CHAP = 500      # Chapman bada hai (~20k), isliye kam
SEED        = 42
RETRAIN_FAIR = False   # True = optional section 9 chalega (multi-seed + same-recipe baseline). ~35 min/run on RTX 3050.

CLASSES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
LEADS   = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

## 1. Data load (same normalisation jo training mein thi: per-lead mean/std, sirf train set se)

In [ ]:
X_train = np.load(PTB_DIR / 'X_train.npy', mmap_mode='r')
MEAN = np.asarray(X_train.mean(axis=(0, 1), keepdims=True), dtype=np.float32)          # (1,1,12)
STD  = np.asarray(X_train.std(axis=(0, 1), keepdims=True), dtype=np.float32) + 1e-8
n_train = len(X_train); del X_train

def prep_ptb(name):   # (N,1000,12) -> normalised (N,12,1000)
    X = np.load(PTB_DIR / f'X_{name}.npy')
    return np.ascontiguousarray(np.transpose((X - MEAN) / STD, (0, 2, 1))).astype(np.float32), np.load(PTB_DIR / f'y_{name}.npy').astype(int)

X_val, y_val   = prep_ptb('val')
X_test, y_test = prep_ptb('test')
X_chap = ((np.load(CHAP_DIR / 'X_chapman.npy') - MEAN.reshape(1, 12, 1)) / STD.reshape(1, 12, 1)).astype(np.float32)
y_chap = np.load(CHAP_DIR / 'y_chapman.npy').astype(int)

print(f'PTB-XL  train {n_train} | val {len(y_val)} | test {len(y_test)}')
print(f'Chapman external {len(y_chap)}')
print(pd.DataFrame({'PTB-XL test %': (100 * y_test.mean(0)).round(1), 'Chapman %': (100 * y_chap.mean(0)).round(1),
                    'PTB-XL test n': y_test.sum(0), 'Chapman n': y_chap.sum(0)}, index=CLASSES).to_string())

## 2. Models

**Baselines (purane / standard models):** Simple 1D-CNN, Inception1D (InceptionTime, Fawaz 2020 / Strodthoff 2021), SE-Inception1D, SE-Temporal-Inception1D, Inception+Transformer.

**Proposed — MorphoWaveNet-v2 (0.72M params).** Paanch components, har ek ek specific problem ke liye:

1. **Morphological derivative layer** — har lead ka V, dV/dt, d²V/dt² (slope = QRS sharpness, curvature = ST/T shape).
2. **Lead-preserving grouped stem** — pehli layer mein har lead alag process hoti hai (purane models pehli conv mein hi 12 leads mix kar dete hain).
3. **Cross-lead spatial attention** — 12×12 attention, model khud decide karta hai ki is ECG ke liye kaunsi leads important hain (aur yeh humein dikhta bhi hai).
4. **Morpho multi-scale kernels + SE** — kernel sizes ECG waves se match: 390 ms (T), 150 ms (P), 90 ms (QRS), 50 ms (onsets).
5. **BiGRU + 3-head wave-attention pooling** — global average pooling ki jagah 3 attention heads (P/QRS/T jaisa) jo batate hain ECG ke kis hisse ko dekha.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------- Baseline 1: Simple 1D-CNN ----------------
class SimpleECGCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv1d(12, 32, 7, 2, 3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(nn.Conv1d(32, 64, 5, 2, 2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(nn.Conv1d(64, 128, 3, 1, 1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        return self.fc(self.conv3(self.conv2(self.conv1(x))).squeeze(-1))

# ---------------- Baselines 2-4: Inception1D family ----------------
class SEBlock1D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.excite = nn.Sequential(nn.Linear(ch, ch // r, bias=False), nn.ReLU(),
                                    nn.Linear(ch // r, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        return x * self.excite(x.mean(dim=-1)).unsqueeze(-1)

class TemporalAttention1D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.attn = nn.Sequential(nn.Conv1d(ch, ch // 2, 1), nn.Tanh(), nn.Conv1d(ch // 2, 1, 1))
    def forward(self, x):
        return (x * torch.softmax(self.attn(x), dim=-1)).sum(dim=-1)

class InceptionModule(nn.Module):
    def __init__(self, in_ch, nf=32, bottleneck=32, ks=(9, 19, 39), use_se=False):
        super().__init__()
        self.bottleneck = nn.Conv1d(in_ch, bottleneck, 1, bias=False)
        self.convs = nn.ModuleList([nn.Conv1d(bottleneck, nf, k, padding='same', bias=False) for k in ks])
        self.pool_branch = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1), nn.Conv1d(in_ch, nf, 1, bias=False))
        self.bn = nn.BatchNorm1d(nf * (len(ks) + 1))
        self.se = SEBlock1D(nf * (len(ks) + 1)) if use_se else nn.Identity()
    def forward(self, x):
        z = self.bottleneck(x)
        out = torch.cat([c(z) for c in self.convs] + [self.pool_branch(x)], dim=1)
        return self.se(torch.relu(self.bn(out)))

class InceptionNet(nn.Module):
    def __init__(self, in_ch=12, n_classes=5, nf=32, depth=6, use_se=False, use_temp_attn=False):
        super().__init__()
        out_ch = nf * 4
        self.blocks = nn.ModuleList([InceptionModule(in_ch if d == 0 else out_ch, nf, use_se=use_se) for d in range(depth)])
        self.shortcuts = nn.ModuleList([nn.Sequential(nn.Conv1d(in_ch if i == 0 else out_ch, out_ch, 1, bias=False),
                                                      nn.BatchNorm1d(out_ch)) for i in range(depth // 3)])
        self.temporal_attn = TemporalAttention1D(out_ch) if use_temp_attn else None
        self.fc = nn.Linear(out_ch, n_classes)
    def forward(self, x):
        res, r = x, 0
        for d, blk in enumerate(self.blocks):
            x = blk(x)
            if d % 3 == 2:
                x = torch.relu(x + self.shortcuts[r](res)); res, r = x, r + 1
        feat = self.temporal_attn(x) if self.temporal_attn is not None else x.mean(dim=-1)
        return self.fc(feat)

# ---------------- Baseline 5: Inception + Transformer ----------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class InceptionTransformer(nn.Module):
    def __init__(self, in_ch=12, n_classes=5, nf=32):
        super().__init__()
        out_ch = nf * 4
        self.inc1 = InceptionModule(in_ch, nf); self.inc2 = InceptionModule(out_ch, nf); self.inc3 = InceptionModule(out_ch, nf)
        self.shortcut = nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch))
        self.pos_enc = PositionalEncoding(out_ch, max_len=200)
        enc_layer = nn.TransformerEncoderLayer(d_model=out_ch, nhead=4, dim_feedforward=256, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.fc = nn.Linear(out_ch, n_classes)
    def forward(self, x):
        res = self.shortcut(x)
        x = torch.relu(self.inc3(self.inc2(self.inc1(x))) + res)
        x = F.adaptive_avg_pool1d(x, 125).transpose(1, 2)
        return self.fc(self.transformer(self.pos_enc(x)).mean(dim=1))

# ---------------- PROPOSED: MorphoWaveNet (v1, v2) ----------------
class SEBlock1d(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                nn.Linear(channels, channels // reduction, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).unsqueeze(-1)

class MorphoSEBlock1d(nn.Module):
    """Multi-scale kernels matched to ECG waves: 390 ms (T), 150 ms (P), 90 ms (QRS), 50 ms (sharp onsets) + SE."""
    def __init__(self, ni, nb_filters=26, kss=(39, 15, 9, 5), bottleneck=32):
        super().__init__()
        self.bottleneck = nn.Conv1d(ni, bottleneck, 1, bias=False)
        self.convs = nn.ModuleList([nn.Conv1d(bottleneck, nb_filters, k, padding=k // 2, bias=False) for k in kss])
        self.pool_conv = nn.Sequential(nn.MaxPool1d(3, 1, 1), nn.Conv1d(ni, nb_filters, 1, bias=False))
        out_ch = (len(kss) + 1) * nb_filters
        self.bn = nn.BatchNorm1d(out_ch)
        self.se = SEBlock1d(out_ch, reduction=8)
    def forward(self, x):
        b = self.bottleneck(x)
        return self.se(F.relu(self.bn(torch.cat([c(b) for c in self.convs] + [self.pool_conv(x)], dim=1))))

class Shortcut1d(nn.Module):
    def __init__(self, ni, nf):
        super().__init__()
        self.conv = nn.Conv1d(ni, nf, 1, bias=False); self.bn = nn.BatchNorm1d(nf)
    def forward(self, inp, out):
        return F.relu(out + self.bn(self.conv(inp)))

class WaveAttentionPool1d(nn.Module):
    """3 learned attention heads over time (P / QRS / T-like) + global max pool."""
    def __init__(self, in_channels, n_heads=3):
        super().__init__()
        self.attn_conv = nn.Sequential(nn.Conv1d(in_channels, in_channels // 2, 5, padding=2, bias=False), nn.Tanh(),
                                       nn.Conv1d(in_channels // 2, n_heads, 1, bias=True))
    def forward(self, x, return_weights=False):
        w = F.softmax(self.attn_conv(x).float(), dim=-1).to(x.dtype)
        out = torch.cat([torch.bmm(w, x.transpose(1, 2)).flatten(1), F.adaptive_max_pool1d(x, 1).flatten(1)], dim=1)
        return (out, w) if return_weights else out

class MorphoWaveNetV1(nn.Module):
    def __init__(self, c_in=12, n_out=5, depth=6, nb_filters=26, kss=(39, 15, 9, 5), bottleneck=32):
        super().__init__()
        stem_ch = c_in * 4
        self.lead_stem = nn.Sequential(
            nn.Conv1d(c_in, stem_ch, 7, padding=3, groups=c_in, bias=False), nn.BatchNorm1d(stem_ch), nn.ReLU(inplace=True),
            nn.Conv1d(stem_ch, stem_ch, 1, bias=False), nn.BatchNorm1d(stem_ch), nn.ReLU(inplace=True))
        nf = (len(kss) + 1) * nb_filters
        self.blocks = nn.ModuleList([MorphoSEBlock1d(stem_ch if d == 0 else nf, nb_filters, kss, bottleneck) for d in range(depth)])
        self.shortcuts = nn.ModuleList([Shortcut1d(stem_ch if d == 0 else nf, nf) for d in range(depth // 3)])
        self.wave_pool = WaveAttentionPool1d(nf, n_heads=3)
        self.head = nn.Sequential(nn.BatchNorm1d(4 * nf), nn.Dropout(0.3), nn.Linear(4 * nf, 160), nn.ReLU(inplace=True),
                                  nn.BatchNorm1d(160), nn.Dropout(0.5), nn.Linear(160, n_out))
    def forward(self, x):
        x = self.lead_stem(x); res = x
        for d, blk in enumerate(self.blocks):
            x = blk(x)
            if d % 3 == 2:
                x = self.shortcuts[d // 3](res, x); res = x
        return self.head(self.wave_pool(x))

class MorphoDerivativeLayer(nn.Module):
    """12 -> 36 channels: voltage V, slope dV/dt, curvature d2V/dt2 for every lead."""
    def forward(self, x):
        dx1 = F.pad(x[:, :, 1:] - x[:, :, :-1], (1, 0))
        dx2 = F.pad(dx1[:, :, 1:] - dx1[:, :, :-1], (1, 0))
        return torch.cat([x, dx1, dx2], dim=1)

class LeadSpatialAttention(nn.Module):
    """12x12 self-attention between leads -> per-lead gate (which leads matter for this ECG)."""
    def __init__(self, n_leads=12, ch_per_lead=4, heads=2):
        super().__init__()
        self.n_leads, self.ch_per_lead = n_leads, ch_per_lead
        self.attn = nn.MultiheadAttention(embed_dim=ch_per_lead, num_heads=heads, batch_first=True)
        self.norm = nn.LayerNorm(ch_per_lead)
        self.gate = nn.Sequential(nn.Linear(ch_per_lead, ch_per_lead), nn.Sigmoid())
    def lead_weights(self, x):
        B, C, T = x.shape
        d = x.view(B, self.n_leads, self.ch_per_lead, T).mean(dim=-1)
        a, _ = self.attn(d, d, d)
        return self.gate(self.norm(d + a))                      # (B, 12, ch_per_lead)
    def forward(self, x):
        B, C, T = x.shape
        w = self.lead_weights(x).unsqueeze(-1)
        return (x.view(B, self.n_leads, self.ch_per_lead, T) * w).reshape(B, C, T)

class MorphoWaveNetV2(nn.Module):
    def __init__(self, c_in=12, n_out=5, depth=6, nb_filters=26, kss=(39, 15, 9, 5), bottleneck=32):
        super().__init__()
        self.c_in = c_in
        self.deriv = MorphoDerivativeLayer()
        ch_per_lead = 4; stem_ch = c_in * ch_per_lead
        self.lead_stem = nn.Sequential(nn.Conv1d(c_in * 3, stem_ch, 7, padding=3, groups=c_in, bias=False),
                                       nn.BatchNorm1d(stem_ch), nn.ReLU(inplace=True))
        self.lead_attn = LeadSpatialAttention(n_leads=c_in, ch_per_lead=ch_per_lead, heads=2)
        self.lead_mix = nn.Sequential(nn.Conv1d(stem_ch, stem_ch, 1, bias=False), nn.BatchNorm1d(stem_ch), nn.ReLU(inplace=True))
        nf = (len(kss) + 1) * nb_filters
        self.blocks = nn.ModuleList([MorphoSEBlock1d(stem_ch if d == 0 else nf, nb_filters, kss, bottleneck) for d in range(depth)])
        self.shortcuts = nn.ModuleList([Shortcut1d(stem_ch if d == 0 else nf, nf) for d in range(depth // 3)])
        self.temporal_gru = nn.GRU(input_size=nf, hidden_size=nf // 2, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.gru_norm = nn.LayerNorm(nf)
        self.wave_pool = WaveAttentionPool1d(nf, n_heads=3)
        self.head = nn.Sequential(nn.BatchNorm1d(4 * nf), nn.Dropout(0.3), nn.Linear(4 * nf, 160), nn.ReLU(inplace=True),
                                  nn.BatchNorm1d(160), nn.Dropout(0.5), nn.Linear(160, n_out))
    def _stem(self, x):
        x36 = self.deriv(x); B, _, T = x36.shape
        return self.lead_stem(x36.view(B, 3, self.c_in, T).transpose(1, 2).reshape(B, 3 * self.c_in, T))
    def forward(self, x, return_attn=False):
        x = self.lead_mix(self.lead_attn(self._stem(x))); res = x
        for d, blk in enumerate(self.blocks):
            x = blk(x)
            if d % 3 == 2:
                x = self.shortcuts[d // 3](res, x); res = x
        s = x.transpose(1, 2)
        with torch.autocast(device_type=x.device.type, enabled=False):
            g, _ = self.temporal_gru(s.float()); s = self.gru_norm(s.float() + g)
        x = s.transpose(1, 2)
        if return_attn:
            feat, w = self.wave_pool(x, return_weights=True)
            return self.head(feat), w
        return self.head(self.wave_pool(x))

## 3. Saare checkpoints load + predictions (cached)

In [ ]:
# (display name, model, checkpoint, inference mode, is_proposed)
# 'full' = poora 10 s ek saath (jaise in models ko train kiya tha), 'tta' = 2.5 s sliding windows, max over windows
MODEL_SPECS = [
    ('Simple 1D-CNN',            lambda: SimpleECGCNN(),                                   'best_simple_cnn.pth',             'full', False),
    ('Inception1D',              lambda: InceptionNet(),                                   'inception_seed42.pt',             'full', False),
    ('SE-Inception1D',           lambda: InceptionNet(use_se=True),                        'se_inception_seed42.pt',          'full', False),
    ('SE-Temporal-Inception1D',  lambda: InceptionNet(use_se=True, use_temp_attn=True),    'se_temporal_inception_seed42.pt', 'full', False),
    ('Inception+Transformer',    lambda: InceptionTransformer(),                           'inc_transformer_seed42.pt',       'full', False),
    ('MorphoWaveNet-v1 (ours)',  lambda: MorphoWaveNetV1(),                                'morpho_wavenet_best.pt',          'tta',  True),
    ('MorphoWaveNet-v2 (ours)',  lambda: MorphoWaveNetV2(),                                'morpho_wavenet_v2_best.pt',       'tta',  True),
]
PROPOSED = 'MorphoWaveNet-v2 (ours)'
BASELINE = 'Inception1D'

@torch.no_grad()
def predict(model, X, mode, bs=128, win=250, stride=125):
    model.eval(); out = []
    L = X.shape[-1]; starts = list(range(0, L - win + 1, stride))
    if starts[-1] != L - win: starts.append(L - win)
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(DEVICE)
        if mode == 'full':
            p = torch.sigmoid(model(xb))
        else:
            B = len(xb); w = torch.stack([xb[:, :, s:s + win] for s in starts], 1).reshape(B * len(starts), 12, win)
            p = torch.sigmoid(model(w)).view(B, len(starts), -1).max(1).values
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)

PRED, PARAMS, MODELS = {}, {}, {}
for name, ctor, ck, mode, _ in MODEL_SPECS:
    m = ctor(); m.load_state_dict(torch.load(CKPT_DIR / ck, map_location='cpu')); m.to(DEVICE).eval()
    MODELS[name] = m; PARAMS[name] = sum(p.numel() for p in m.parameters())
    cache = OUT / 'preds' / f'{ck}.npz'
    if cache.exists() and np.load(cache)['test'].shape[0] == len(y_test) and np.load(cache)['chap'].shape[0] == len(y_chap):
        z = np.load(cache); PRED[name] = {k: z[k] for k in ['val', 'test', 'chap']}; src = 'cache'
    else:
        t0 = time.time()
        PRED[name] = {'val': predict(m, X_val, mode), 'test': predict(m, X_test, mode), 'chap': predict(m, X_chap, mode)}
        np.savez_compressed(cache, **PRED[name]); src = f'{time.time() - t0:.0f}s'
    print(f'{name:26s} params={PARAMS[name]:>9,}  test macro-AUROC={roc_auc_score(y_test, PRED[name]["test"], average="macro"):.4f}  ({src})')

**Sanity check:** training notebooks ne jo test probabilities save ki thi, unse match karo (same model + same data ⇒ same numbers).

In [ ]:
for f, name in [('morpho_wavenet_v2_test_probs.npy', PROPOSED), ('se_temp_test_probs.npy', 'SE-Temporal-Inception1D'),
                ('inception_test_probs.npy', 'Inception1D')]:
    if Path(f).exists():
        old = np.load(f)
        if old.shape == PRED[name]['test'].shape:
            print(f'{name:26s} vs {f:34s}: max |diff| = {np.abs(old - PRED[name]["test"]).max():.2e}  '
                  f'(old AUROC {roc_auc_score(y_test, old, average="macro"):.4f})')
        else:
            print(f'{f}: shape {old.shape} != {PRED[name]["test"].shape} (skip)')

## 4. Metric helpers (thresholds sirf fold 9 pe tune, test pe apply; bootstrap = same resamples sab models ke liye → paired)

In [ ]:
def auc_fast(y, s):                       # exact AUROC with ties (rank formula), same as sklearn
    r = rankdata(s); npos = y.sum(); nneg = len(y) - npos
    return (r[y == 1].sum() - npos * (npos + 1) / 2) / (npos * nneg)

def macro_auc(Y, P): return np.mean([auc_fast(Y[:, c], P[:, c]) for c in range(Y.shape[1])])

def tune_thresholds(Y, P, grid=np.arange(0.05, 0.96, 0.01)):
    return np.array([grid[int(np.argmax([f1_score(Y[:, c], P[:, c] >= t, zero_division=0) for t in grid]))] for c in range(Y.shape[1])])

def macro_f1(Y, Yhat):
    tp = (Y & Yhat).sum(0); fp = (~Y.astype(bool) & Yhat.astype(bool)).sum(0); fn = (Y.astype(bool) & ~Yhat.astype(bool)).sum(0)
    return np.mean(2 * tp / np.maximum(2 * tp + fp + fn, 1))

def boot_indices(Y, n, seed=SEED):
    rng = np.random.default_rng(seed); out = []
    while len(out) < n:
        i = rng.integers(0, len(Y), len(Y))
        s = Y[i].sum(0)
        if (s > 0).all() and (s < len(Y)).all(): out.append(i)
    return out

BOOT_TEST = boot_indices(y_test, N_BOOT)
BOOT_CHAP = boot_indices(y_chap, N_BOOT_CHAP, seed=SEED + 1)
THR = {n: tune_thresholds(y_val, PRED[n]['val']) for n in PRED}

def evaluate(split, Y, boots):
    rows, BA = [], {}
    for n in PRED:
        P = PRED[n][split]; Yhat = (P >= THR[n]).astype(int)
        ba = np.array([macro_auc(Y[i], P[i]) for i in boots]); BA[n] = ba
        bf = np.array([macro_f1(Y[i], Yhat[i]) for i in boots])
        rows.append({'Model': n, 'Params': PARAMS[n],
                     'Macro-AUROC': macro_auc(Y, P), 'AUROC 95% CI': f'[{np.percentile(ba, 2.5):.4f}, {np.percentile(ba, 97.5):.4f}]',
                     'Macro-AUPRC': average_precision_score(Y, P, average='macro'),
                     'Macro-F1': macro_f1(Y, Yhat), 'F1 95% CI': f'[{np.percentile(bf, 2.5):.4f}, {np.percentile(bf, 97.5):.4f}]'})
    return pd.DataFrame(rows).set_index('Model'), BA
print('thresholds (fold 9):'); print(pd.DataFrame(THR, index=CLASSES).T.round(2).to_string())

## RESULT 1 — PTB-XL test set (fold 10, n = 2163)

In [ ]:
t0 = time.time()
TAB_PTB, BOOT_AUC_PTB = evaluate('test', y_test, BOOT_TEST)
print(f'(bootstrap {time.time() - t0:.0f}s)\n')
show = TAB_PTB.copy()
for c in ['Macro-AUROC', 'Macro-AUPRC', 'Macro-F1']: show[c] = show[c].round(4)
show['Params'] = show['Params'].map(lambda v: f'{v / 1e6:.3f}M')
print(show.to_string())
show.to_csv(OUT / 'table1_ptbxl_test.csv')

## RESULT 2 — Published papers se comparison (same task: PTB-XL superdiagnostic, fold 10, macro-AUROC)
Published numbers papers se as-is liye gaye hain (unke models humne re-run nahi kiye). Hamari rows upar wale evaluation se aa rahi hain.

In [ ]:
LIT = [  # (method, source, macro-AUROC, note)
    ('Wavelet + NN',              'Strodthoff et al., IEEE JBHI 2021', 0.874, 'feature-based'),
    ('inception1d',               'Strodthoff et al., IEEE JBHI 2021', 0.921, ''),
    ('lstm_bidir',                'Strodthoff et al., IEEE JBHI 2021', 0.921, ''),
    ('fcn_wang',                  'Strodthoff et al., IEEE JBHI 2021', 0.925, ''),
    ('lstm',                      'Strodthoff et al., IEEE JBHI 2021', 0.927, ''),
    ('xresnet1d101',              'Strodthoff et al., IEEE JBHI 2021', 0.928, ''),
    ('resnet1d_wang',             'Strodthoff et al., IEEE JBHI 2021', 0.930, 'best single model in benchmark'),
    ('Ensemble of all models',    'Strodthoff et al., IEEE JBHI 2021', 0.934, 'ensemble'),
    ('CWT-ViT-KAN',               'Mert et al., arXiv 2609.12803 (2026)', 0.874, ''),
    ('xLSTM',                     'Mert et al., arXiv 2609.12803 (2026)', 0.910, ''),
    ('1D-ResNet18',               'Mert et al., arXiv 2609.12803 (2026)', 0.922, ''),
    ('Bidirectional Mamba',       'Mert et al., arXiv 2609.12803 (2026)', 0.923, ''),
    ('ECGFounder (fine-tuned)',   'Mert et al., arXiv 2609.12803 (2026)', 0.930, 'foundation model, pretrained on 10M+ ECGs'),
    ('Stacking ensemble (5 models)', 'Mert et al., arXiv 2609.12803 (2026)', 0.936, 'ensemble incl. ECGFounder'),
]
lit = pd.DataFrame(LIT, columns=['Method', 'Source', 'Macro-AUROC', 'Note'])
ours = pd.DataFrame([{'Method': n, 'Source': 'This work', 'Macro-AUROC': round(TAB_PTB.loc[n, 'Macro-AUROC'], 4),
                      'Note': f'{PARAMS[n] / 1e6:.2f}M params, no pretraining, single model'} for n in PRED if 'ours' in n])
TAB_LIT = pd.concat([lit, ours]).sort_values('Macro-AUROC').reset_index(drop=True)
print(TAB_LIT.to_string())
TAB_LIT.to_csv(OUT / 'table2_literature_comparison.csv', index=False)

singles = TAB_LIT[~TAB_LIT.Note.str.contains('ensemble') & (TAB_LIT.Source != 'This work')]
best_single = singles.loc[singles['Macro-AUROC'].idxmax()]
print(f"\nBest published SINGLE model: {best_single.Method} ({best_single.Source}) = {best_single['Macro-AUROC']:.3f}")
print(f"{PROPOSED}: {TAB_PTB.loc[PROPOSED, 'Macro-AUROC']:.4f}  (95% CI {TAB_PTB.loc[PROPOSED, 'AUROC 95% CI']})")

## RESULT 3 — Statistical significance (paired bootstrap, PTB-XL test)
Har resample pe ΔAUROC = (MorphoWaveNet-v2 − baseline). Agar 95% CI mein 0 nahi aata ⇒ improvement significant. p = two-sided bootstrap p-value.

In [ ]:
def paired(BA, a, b):
    d = BA[a] - BA[b]
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), max(p, 1 / len(d))

rows = []
for n in PRED:
    if n == PROPOSED: continue
    m, lo, hi, p = paired(BOOT_AUC_PTB, PROPOSED, n)
    rows.append({'Proposed vs': n, 'ΔAUROC': round(m, 4), '95% CI': f'[{lo:+.4f}, {hi:+.4f}]', 'p-value': round(p, 4),
                 'Significant (p<0.05)?': 'YES' if (lo > 0 or hi < 0) else 'no'})
TAB_SIG = pd.DataFrame(rows).set_index('Proposed vs')
print(TAB_SIG.to_string()); TAB_SIG.to_csv(OUT / 'table3_significance.csv')

## RESULT 4 — Per-class AUROC (PTB-XL test)

In [ ]:
PC = pd.DataFrame({n: [auc_fast(y_test[:, c], PRED[n]['test'][:, c]) for c in range(5)] for n in PRED}, index=CLASSES).T.round(4)
PC['Macro'] = PC.mean(1).round(4)
print(PC.to_string()); PC.to_csv(OUT / 'table4_per_class_auroc.csv')

fig, ax = plt.subplots(figsize=(11, 4.5)); names = list(PRED); w = 0.8 / len(names); x = np.arange(5)
for k, n in enumerate(names):
    ax.bar(x + (k - len(names) / 2 + 0.5) * w, PC.loc[n, CLASSES], w, label=n,
           color='C3' if n == PROPOSED else ('C1' if 'ours' in n else None), edgecolor='k' if n == PROPOSED else 'none')
ax.set_xticks(x); ax.set_xticklabels(CLASSES); ax.set_ylim(min(0.85, PC[CLASSES].values.min() - 0.01), max(0.97, PC[CLASSES].values.max() + 0.01)); ax.set_ylabel('AUROC (PTB-XL test)')
ax.set_xlabel('Diagnostic superclass'); ax.set_title('Per-class AUROC'); ax.legend(fontsize=8, ncol=4, loc='upper left')
fig.tight_layout(); fig.savefig(OUT / 'fig_per_class_auroc.png', dpi=200); plt.show()

## RESULT 5 — External validation: Chapman-Shaoxing (alag hospital, alag machine, zero retraining)
Same weights, same fold-9 thresholds. Note: Chapman mein MI sirf ~0.6% hai, isliye MI ka AUPRC/F1 low aana expected hai.

In [ ]:
t0 = time.time()
TAB_CHAP, BOOT_AUC_CHAP = evaluate('chap', y_chap, BOOT_CHAP)
TAB_CHAP['PTB-XL AUROC'] = TAB_PTB['Macro-AUROC']
TAB_CHAP['Drop (PTB→Chapman)'] = TAB_CHAP['PTB-XL AUROC'] - TAB_CHAP['Macro-AUROC']
print(f'(bootstrap {time.time() - t0:.0f}s)\n')
show = TAB_CHAP.drop(columns=['Params']).copy()
for c in ['Macro-AUROC', 'Macro-AUPRC', 'Macro-F1', 'PTB-XL AUROC', 'Drop (PTB→Chapman)']: show[c] = show[c].round(4)
print(show.to_string()); show.to_csv(OUT / 'table5_chapman_external.csv')

PCC = pd.DataFrame({n: [auc_fast(y_chap[:, c], PRED[n]['chap'][:, c]) for c in range(5)] for n in PRED}, index=CLASSES).T.round(4)
print('\nChapman per-class AUROC:'); print(PCC.to_string()); PCC.to_csv(OUT / 'table5b_chapman_per_class.csv')

rows = []
for n in PRED:
    if n == PROPOSED: continue
    m, lo, hi, p = paired(BOOT_AUC_CHAP, PROPOSED, n)
    rows.append({'Proposed vs': n, 'ΔAUROC (Chapman)': round(m, 4), '95% CI': f'[{lo:+.4f}, {hi:+.4f}]', 'p-value': round(p, 4),
                 'Significant?': 'YES' if (lo > 0 or hi < 0) else 'no'})
TAB_SIG_CHAP = pd.DataFrame(rows).set_index('Proposed vs'); print('\n'); print(TAB_SIG_CHAP.to_string())
TAB_SIG_CHAP.to_csv(OUT / 'table5c_chapman_significance.csv')

## RESULT 6 — Main comparison figure + efficiency (parameters vs AUROC)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
names = list(PRED); x = np.arange(len(names))
for off, (tab, lab) in zip([-0.2, 0.2], [(TAB_PTB, 'PTB-XL test (in-domain)'), (TAB_CHAP, 'Chapman (external)')]):
    mu = tab['Macro-AUROC'].values
    ci = np.array([[float(v) for v in s.strip('[]').split(',')] for s in tab['AUROC 95% CI']])
    axs[0].bar(x + off, mu, 0.4, yerr=[mu - ci[:, 0], ci[:, 1] - mu], capsize=3, label=lab)
axs[0].set_xticks(x); axs[0].set_xticklabels([n.replace(' (ours)', '\n(ours)') for n in names], rotation=35, ha='right', fontsize=9)
lo_ = min(TAB_PTB['Macro-AUROC'].min(), TAB_CHAP['Macro-AUROC'].min()); axs[0].set_ylim(min(0.85, lo_ - 0.02), max(0.95, TAB_PTB['Macro-AUROC'].max() + 0.02)); axs[0].set_ylabel('Macro-AUROC (95% CI)'); axs[0].set_title('In-domain vs external'); axs[0].legend(loc='upper left')

for n in names:
    axs[1].scatter(PARAMS[n] / 1e6, TAB_PTB.loc[n, 'Macro-AUROC'], s=140 if 'ours' in n else 60,
                   c='C3' if 'ours' in n else 'C0', marker='*' if 'ours' in n else 'o', zorder=3)
    axs[1].annotate(n, (PARAMS[n] / 1e6, TAB_PTB.loc[n, 'Macro-AUROC']), fontsize=8, xytext=(5, -10), textcoords='offset points')
for val, lab, ls in [(0.928, 'xresnet1d101 (Strodthoff 2021)', ':'), (0.930, 'ECGFounder fine-tuned, 10M+ ECG pretraining', '--'),
                     (0.934, 'Strodthoff 2021 ensemble', '-.')]:
    axs[1].axhline(val, color='gray', ls=ls, lw=1); axs[1].text(0.02, val + 0.0005, lab, fontsize=8, color='gray')
axs[1].set_xlabel('Parameters (millions)'); axs[1].set_ylabel('PTB-XL test macro-AUROC'); axs[1].set_title('Accuracy vs model size')
axs[1].set_ylim(min(0.90, TAB_PTB['Macro-AUROC'].min() - 0.005), max(0.94, TAB_PTB['Macro-AUROC'].max() + 0.005))
fig.tight_layout(); fig.savefig(OUT / 'fig_main_comparison.png', dpi=200); plt.show()

## RESULT 7 — Explainability (built-in, koi post-hoc Grad-CAM nahi chahiye)
(a) **Lead attention**: har class ke liye model ne average kaunsi leads ko zyada weight diya (PTB-XL test, true-positive records).
(b) **Wave-attention heads**: ek MI patient ke ECG pe 3 heads ne kahan dekha.

In [ ]:
v2 = MODELS[PROPOSED]

@torch.no_grad()
def lead_gates(X, bs=128, win=250, stride=125):
    out = []; starts = list(range(0, X.shape[-1] - win + 1, stride))
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(DEVICE); B = len(xb)
        w = torch.stack([xb[:, :, s:s + win] for s in starts], 1).reshape(B * len(starts), 12, win)
        g = v2.lead_attn.lead_weights(v2._stem(w)).mean(-1)            # (B*nw, 12)
        out.append(g.view(B, len(starts), 12).mean(1).cpu().numpy())
    return np.concatenate(out)

G = lead_gates(X_test)
LG = pd.DataFrame({c: G[(y_test[:, k] == 1) & (PRED[PROPOSED]['test'][:, k] >= THR[PROPOSED][k])].mean(0)
                   for k, c in enumerate(CLASSES)}, index=LEADS).T
LGn = LG.div(LG.mean(1), axis=0)                                          # relative importance (1.0 = average lead)
print('Relative lead importance per class (true positives, >1 = zyada important):')
print(LGn.round(3).to_string())
print('\nTop-3 leads per class:', {c: list(LGn.loc[c].sort_values(ascending=False).index[:3]) for c in CLASSES})
LGn.to_csv(OUT / 'table7_lead_importance.csv')

fig, ax = plt.subplots(figsize=(10, 3.6))
im = ax.imshow(LGn.values, cmap='RdBu_r', vmin=2 - LGn.values.max(), vmax=LGn.values.max(), aspect='auto')
ax.set_xticks(range(12)); ax.set_xticklabels(LEADS); ax.set_yticks(range(5)); ax.set_yticklabels(CLASSES)
for i in range(5):
    for j in range(12): ax.text(j, i, f'{LGn.values[i, j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_xlabel('Lead'); ax.set_title('MorphoWaveNet-v2: learned lead importance per class (PTB-XL test)')
fig.colorbar(im, ax=ax, label='relative weight'); fig.tight_layout(); fig.savefig(OUT / 'fig_lead_importance.png', dpi=200); plt.show()

In [ ]:
mi = CLASSES.index('MI')
cand = np.where((y_test[:, mi] == 1) & (PRED[PROPOSED]['test'][:, mi] >= THR[PROPOSED][mi]))[0]
idx = int(cand[np.argmax(PRED[PROPOSED]['test'][cand, mi])]) if len(cand) else 0
x = torch.from_numpy(X_test[idx:idx + 1]).to(DEVICE)
starts = list(range(0, 1000 - 250 + 1, 125))
with torch.no_grad():
    probs = [torch.sigmoid(v2(x[:, :, s:s + 250]))[0, mi].item() for s in starts]
    s0 = starts[int(np.argmax(probs))]
    _, attn = v2(x[:, :, s0:s0 + 250], return_attn=True)
attn = attn[0].cpu().numpy(); t = np.arange(1000) / 100
sig = X_test[idx, 1] * STD[0, 0, 1] + MEAN[0, 0, 1]

fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
axs[0].plot(t, sig, 'k', lw=0.9); axs[0].axvspan(s0 / 100, (s0 + 250) / 100, color='C3', alpha=0.12, label='window used for decision')
axs[0].set_ylabel('Lead II (mV)'); axs[0].legend(loc='upper right')
axs[0].set_title(f'Test record #{idx} | true: {[c for c, v in zip(CLASSES, y_test[idx]) if v]} | '
                 f'P(MI) = {PRED[PROPOSED]["test"][idx, mi]:.2f}')
for h in range(3):
    axs[1].plot(t[s0:s0 + 250], attn[h] / attn[h].max(), label=f'attention head {h + 1}')
axs[1].set_xlabel('Time (s)'); axs[1].set_ylabel('attention (norm.)'); axs[1].legend(loc='upper right', fontsize=8)
fig.tight_layout(); fig.savefig(OUT / 'fig_wave_attention_MI.png', dpi=200); plt.show()

## RESULT 8 — Purane models ki problem → hamara solution → evidence
Saare numbers upar ke evaluation se automatically bhare gaye hain. Ablation rows (Morpho-Inception) `chap.ipynb` ke training logs se hain (unka checkpoint save nahi hua tha).

In [ ]:
A = lambda n: TAB_PTB.loc[n, 'Macro-AUROC']
sig_base = TAB_SIG.loc[BASELINE]
best_pub_single = 0.930
PROB = pd.DataFrame([
    ['Pehli conv layer mein 12 leads mix ho jaati hain → lead-specific morphology (e.g. V1–V3 mein MI) kho jaati hai',
     'Lead-preserving grouped stem + cross-lead spatial attention',
     f'Lead importance heatmap (RESULT 7: check MI → II/III/aVF, V1–V4); v1 → v2 AUROC {A("MorphoWaveNet-v1 (ours)"):.4f} → {A(PROPOSED):.4f}'],
    ['Global average pooling se "ECG ke kis hisse mein problem hai" ki info chali jaati hai',
     '3-head wave-attention pooling + BiGRU temporal context',
     f'Overall (saare components milake): Inception1D (GAP) {A(BASELINE):.4f} vs v2 {A(PROPOSED):.4f}; ΔAUROC {sig_base["ΔAUROC"]:+.4f}, 95% CI {sig_base["95% CI"]}, p = {sig_base["p-value"]}'],
    ['Generic kernel sizes (9/19/39) ECG waves se match nahi karte; slope/curvature implicitly seekhna padta hai',
     'Morpho kernels (5/9/15/39 = onset/QRS/P/T) + explicit dV/dt, d²V/dt² input',
     'Training-log ablation (chap.ipynb): Morpho-Inception + TTA 0.9300 → + lead stem/SE/wave-attn (v1) → + derivatives/lead-attn/GRU (v2)'],
    ['Black box: explanation ke liye alag se Grad-CAM / perturbation chahiye',
     'Attention built-in hai: lead weights + wave-attention directly output',
     'RESULT 7 figures (lead importance heatmap, wave-attention on MI patient)'],
    ['High accuracy ke liye bade models / 10M-ECG pretraining / ensembles chahiye',
     f'Single model, {PARAMS[PROPOSED] / 1e6:.2f}M params, sirf PTB-XL pe train, koi pretraining nahi',
     f'{A(PROPOSED):.4f} vs ECGFounder 0.930 (foundation model), xresnet1d101 0.928, Strodthoff ensemble 0.934'],
    ['Doosre hospital/machine pe performance girti hai (domain shift)',
     'Same model, zero retraining, Chapman pe test',
     f'Chapman AUROC {TAB_CHAP.loc[PROPOSED, "Macro-AUROC"]:.4f} (drop {TAB_CHAP.loc[PROPOSED, "Drop (PTB→Chapman)"]:.4f}) vs Inception1D {TAB_CHAP.loc[BASELINE, "Macro-AUROC"]:.4f} (drop {TAB_CHAP.loc[BASELINE, "Drop (PTB→Chapman)"]:.4f})'],
], columns=['Purane models ki problem', 'MorphoWaveNet ka solution', 'Evidence (is notebook se)'])
pd.set_option('display.max_colwidth', 200)
display(PROB) if 'display' in dir() else print(PROB.to_string())
PROB.to_csv(OUT / 'table8_problem_solution_evidence.csv', index=False)

## 9. (Optional) Fair comparison: multi-seed + same training recipe
**Kyun:** saved Inception1D poore 10 s pe bina augmentation/TTA ke train hua tha, jabki MorphoWaveNet ko windowing + augmentation + TTA mila. Reviewer poochega "gain architecture se hai ya training recipe se?" Yeh section Inception1D ko **bilkul same recipe** se aur dono models ko 3 seeds pe train karta hai. `RETRAIN_FAIR = True` karo (RTX 3050 pe ~35 min per run, 6 runs).

In [ ]:
FAIR_SEEDS = [0, 1, 2]

def augment(xb, p=0.5):   # same as MorphoWaveNet training (RRC + time-out, decision per batch)
    B, C, L = xb.shape
    if torch.rand(1).item() < p:
        scale = torch.empty(B, 1, device=xb.device).uniform_(0.6, 1.0); start = torch.rand(B, 1, device=xb.device) * (1 - scale)
        gx = (start + scale * torch.linspace(0, 1, L, device=xb.device)[None]) * 2 - 1
        xb = F.grid_sample(xb[:, :, None], torch.stack([gx, torch.zeros_like(gx)], -1)[:, None], mode='bilinear', align_corners=True)[:, :, 0]
    if torch.rand(1).item() < p:
        t_ = torch.empty(B, 1, device=xb.device).uniform_(0, 0.35); s = torch.rand(B, 1, device=xb.device) * (1 - t_)
        pos = torch.linspace(0, 1, L, device=xb.device)[None]; xb = xb * (~((pos >= s) & (pos < s + t_)))[:, None].to(xb.dtype)
    return xb

def train_same_recipe(model, seed, tag, epochs=50, lr=6e-3, wd=1.5e-2, win=250, patience=12, bs=64, smooth=0.02):
    ck = OUT / f'fair_{tag}_s{seed}.pt'
    torch.manual_seed(seed); np.random.seed(seed); model.to(DEVICE)
    if ck.exists():
        model.load_state_dict(torch.load(ck, map_location=DEVICE)); return model
    Xtr = np.ascontiguousarray(np.transpose((np.load(PTB_DIR / 'X_train.npy') - MEAN) / STD, (0, 2, 1))).astype(np.float32)
    ytr = np.load(PTB_DIR / 'y_train.npy').astype(np.float32)
    Xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr)
    steps = math.ceil(len(Xt) / bs)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, steps_per_epoch=steps, pct_start=0.25)
    best, bad, warm = 0, 0, int(epochs * 0.25)
    for ep in range(1, epochs + 1):
        model.train(); perm = torch.randperm(len(Xt))
        for b in range(steps):
            i = perm[b * bs:(b + 1) * bs]; xb, yb = Xt[i].to(DEVICE), yt[i].to(DEVICE)
            st = torch.randint(0, xb.shape[-1] - win + 1, (len(i),), device=DEVICE)
            xb = augment(torch.gather(xb, 2, (st[:, None] + torch.arange(win, device=DEVICE)[None])[:, None].expand(-1, 12, -1)))
            loss = F.binary_cross_entropy_with_logits(model(xb), yb * (1 - smooth) + smooth / 2)
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); sch.step()
        va = roc_auc_score(y_val, predict(model, X_val, 'tta'), average='macro')
        if va > best + 1e-4: best, bad = va, 0; torch.save(model.state_dict(), ck)
        elif ep > warm: bad += 1
        print(f'  [{tag} s{seed}] ep {ep:02d} val AUROC {va:.4f}{" *" if bad == 0 else ""}')
        if ep > warm and bad >= patience: break
    model.load_state_dict(torch.load(ck, map_location=DEVICE)); return model

if RETRAIN_FAIR:
    fair = []
    for tag, ctor in [('Inception1D_same_recipe', lambda: InceptionNet()), ('MorphoWaveNet-v2', lambda: MorphoWaveNetV2())]:
        for s in FAIR_SEEDS:
            m = train_same_recipe(ctor(), s, tag)
            pt = predict(m, X_test, 'tta'); pc = predict(m, X_chap, 'tta')
            fair.append({'Model': tag, 'seed': s, 'PTB-XL AUROC': roc_auc_score(y_test, pt, average='macro'),
                         'Chapman AUROC': roc_auc_score(y_chap, pc, average='macro')})
            print(fair[-1])
    TAB_FAIR = pd.DataFrame(fair); TAB_FAIR.to_csv(OUT / 'table9_fair_multiseed_runs.csv', index=False)
    summ = TAB_FAIR.groupby('Model')[['PTB-XL AUROC', 'Chapman AUROC']].agg(['mean', 'std']).round(4)
    print(summ.to_string()); summ.to_csv(OUT / 'table9_fair_multiseed_summary.csv')
else:
    print('RETRAIN_FAIR = False -> skipped. (Sir/reviewer ke liye strongly recommended: True karke ek baar raat bhar chala do.)')

## 10. Sab kuch save — `results_final/SUMMARY.md` sir ko bhejne ke liye

In [ ]:
def md_table(df):
    df = df.reset_index() if df.index.name else df
    cols = list(df.columns); lines = ['| ' + ' | '.join(map(str, cols)) + ' |', '|' + '---|' * len(cols)]
    for _, r in df.iterrows(): lines.append('| ' + ' | '.join(f'{v:.4f}' if isinstance(v, float) else str(v) for v in r) + ' |')
    return '\n'.join(lines)

p = TAB_PTB.loc[PROPOSED]; c = TAB_CHAP.loc[PROPOSED]
summary = f'''# MorphoWaveNet-v2 — results summary

**Task:** PTB-XL superdiagnostic (5 classes, multi-label), folds 1-8 train / 9 val / 10 test (n={len(y_test)}). External: Chapman-Shaoxing (n={len(y_chap)}), zero retraining.

## Headline
- PTB-XL test macro-AUROC **{p["Macro-AUROC"]:.4f}** (95% CI {p["AUROC 95% CI"]}), macro-F1 {p["Macro-F1"]:.4f}, {PARAMS[PROPOSED] / 1e6:.2f}M params, single model, no pretraining.
- vs {BASELINE} (same data/split): ΔAUROC {TAB_SIG.loc[BASELINE, "ΔAUROC"]:+.4f}, 95% CI {TAB_SIG.loc[BASELINE, "95% CI"]}, p = {TAB_SIG.loc[BASELINE, "p-value"]}.
- Published: xresnet1d101 0.928, resnet1d_wang 0.930 (Strodthoff 2021), ECGFounder fine-tuned 0.930 (pretrained on 10M+ ECGs), ensembles 0.934-0.936.
- Chapman external macro-AUROC **{c["Macro-AUROC"]:.4f}** (drop {c["Drop (PTB→Chapman)"]:.4f}).

## Table 1 — PTB-XL test
{md_table(TAB_PTB.drop(columns=["Params"]).round(4))}

## Table 2 — Significance (paired bootstrap, {N_BOOT} resamples)
{md_table(TAB_SIG)}

## Table 3 — Literature comparison
{md_table(TAB_LIT)}

## Table 4 — Chapman external
{md_table(TAB_CHAP.drop(columns=["Params"]).round(4))}

## Problem → solution → evidence
{md_table(PROB)}

## Caveats (honest)
- Saare apne models single seed (42) hain. Multi-seed aur same-recipe baseline: section 9 (`RETRAIN_FAIR=True`).
- Literature numbers papers se as-is liye gaye hain; unke models re-run nahi kiye.
- Figures: fig_main_comparison.png, fig_per_class_auroc.png, fig_lead_importance.png, fig_wave_attention_MI.png
'''
(OUT / 'SUMMARY.md').write_text(summary, encoding='utf-8')
json.dump({'ptbxl_test': TAB_PTB.reset_index().to_dict('records'), 'chapman': TAB_CHAP.reset_index().to_dict('records'),
           'significance_ptbxl': TAB_SIG.reset_index().to_dict('records')}, open(OUT / 'results.json', 'w'), indent=1, default=float)
print('Saved to', OUT.resolve()); print(sorted(p_.name for p_ in OUT.iterdir()))